# Building Features EDA — OSM

Dataset:

`data/processed/dataset_city_balanced_eda.csv`

Purpose:

Explore the 7 OpenStreetMap building features in the urban dataset.

Each row is one 500 m × 500 m city patch.

Feature groups covered here:

- **Identification features** — patch_id, city, code, lat, lon
- **Building features (7)** — building_count, pct_apartments, pct_houses, pct_commercial, pct_other, building_coverage, public_space_ratio
- **Entropy features** — entropy_raw, entropy_normalised, entropy_weighted_raw, entropy_weighted_norm, orientation_order_phi

Visualisations:

1. Distribution of building features
2. Correlation matrix — building features vs entropy
3. Pairplot of building features
4. Ranked mean entropy by city
5. Entropy distribution per city ordered by mean slope (flattest → steepest)

## 1 — Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# ── plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="tab10", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

## 2 — Load Dataset

In [ ]:
# Try both path variants — notebook may run from repo root or from notebooks/
DATA_PATH = Path("../data/processed/dataset_city_balanced_eda.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/processed/dataset_city_balanced_eda.csv")

df = pd.read_csv(DATA_PATH)

print(f"Rows   : {len(df)}")
print(f"Columns: {len(df.columns)}")
df.head(3)

## 3 — Identification Features

Columns that identify each patch — not used in modelling, but essential for grouping, mapping, and cross-referencing.

In [ ]:
ID_COLS = ["patch_id", "city", "code", "lat", "lon"]

# keep only those that actually exist in the loaded file
id_cols_present = [c for c in ID_COLS if c in df.columns]

print("Identification columns present:", id_cols_present)
print()
print(f"Cities  : {sorted(df['code'].unique().tolist())}")
print(f"Patches : {len(df)}")
df[id_cols_present].head(10)

## 4 — Building Features

The 7 OSM-derived building features computed by `04_compute_building_features.py`:

| Feature | Description |
|---|---|
| `building_count` | Number of building footprints in the patch |
| `pct_apartments` | Share of buildings tagged as apartments / residential blocks |
| `pct_houses` | Share of buildings tagged as detached / semi-detached houses |
| `pct_commercial` | Share of buildings tagged as commercial / retail / office |
| `pct_other` | Remaining building types not captured above |
| `building_coverage` | Fraction of patch area covered by building footprints |
| `public_space_ratio` | Fraction of patch area covered by parks, grass, and pedestrian zones |

In [ ]:
BUILDING_COLS = [
    "building_count",
    "pct_apartments",
    "pct_houses",
    "pct_commercial",
    "pct_other",
    "building_coverage",
    "public_space_ratio",
]

# keep only columns present in the dataset
building_cols = [c for c in BUILDING_COLS if c in df.columns]

print("Building columns present:", building_cols)
print()
df[building_cols].describe().T.round(4)

## 5 — Entropy Features

Street-orientation entropy features computed by `06_compute_entropy.py` (Boeing 2019 method):

| Feature | Description |
|---|---|
| `entropy_raw` | Shannon entropy of unweighted bearing distribution (nats) |
| `entropy_normalised` | `entropy_raw` divided by log(36) — range 0–1 |
| `entropy_weighted_raw` | Shannon entropy weighted by street-segment length (nats) |
| `entropy_weighted_norm` | `entropy_weighted_raw` divided by log(36) — range 0–1 |
| `orientation_order_phi` | Orientation-order index φ: 1 = perfect grid, 0 = maximum disorder |

In [ ]:
ENTROPY_COLS = [
    "entropy_raw",
    "entropy_normalised",
    "entropy_weighted_raw",
    "entropy_weighted_norm",
    "orientation_order_phi",
]

entropy_cols = [c for c in ENTROPY_COLS if c in df.columns]

print("Entropy columns present:", entropy_cols)
print()
df[entropy_cols].describe().T.round(4)

## 6 — Distribution of Building Features

Histograms + KDE curves for each of the 7 building features, coloured by city.

In [ ]:
n_features = len(building_cols)
n_cols     = 3
n_rows     = int(np.ceil(n_features / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

cities     = sorted(df["code"].unique())
city_pal   = sns.color_palette("tab10", n_colors=len(cities))
city_color = dict(zip(cities, city_pal))

for idx, feat in enumerate(building_cols):
    ax = axes[idx]

    # overall distribution — grey fill
    sns.histplot(
        data=df,
        x=feat,
        bins=30,
        kde=True,
        color="steelblue",
        alpha=0.35,
        ax=ax,
        label="All cities",
    )

    # per-city KDE overlay
    for city in cities:
        subset = df[df["code"] == city][feat].dropna()
        if len(subset) > 5:
            subset.plot.kde(
                ax=ax,
                color=city_color[city],
                linewidth=1.5,
                label=city,
            )

    ax.set_title(feat, fontsize=12, fontweight="bold")
    ax.set_xlabel(feat)
    ax.set_ylabel("Density")

# hide any spare axes
for idx in range(n_features, len(axes)):
    axes[idx].set_visible(False)

# shared legend below the grid
handles = [
    plt.Line2D([0], [0], color=city_color[c], linewidth=2, label=c)
    for c in cities
]
fig.legend(
    handles=handles,
    loc="lower center",
    ncol=len(cities),
    bbox_to_anchor=(0.5, -0.02),
    title="City",
    frameon=False,
)

fig.suptitle("Distribution of Building Features (per city KDE overlay)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7 — Correlation Matrix: Building Features vs Entropy

Pearson correlation between each of the 7 building features (rows) and each entropy feature (columns).

A positive value (warm colour) means the two features tend to increase together.
A negative value (cool colour) means they move in opposite directions.

In [ ]:
if not entropy_cols:
    print("No entropy columns found in the dataset.")
    print("Run 06_compute_entropy.py and reassemble to add entropy features.")
else:
    # cross-correlation: building features as rows, entropy as columns
    cross_corr = (
        df[building_cols + entropy_cols]
        .corr(method="pearson")
        .loc[building_cols, entropy_cols]
    )

    fig, ax = plt.subplots(figsize=(len(entropy_cols) * 2 + 2, len(building_cols) * 1.2 + 1))

    sns.heatmap(
        cross_corr,
        annot=True,
        fmt=".2f",
        cmap="RdBu_r",
        center=0,
        vmin=-1,
        vmax=1,
        linewidths=0.5,
        square=True,
        cbar_kws={"shrink": 0.8, "label": "Pearson r"},
        ax=ax,
    )

    ax.set_title("Pearson Correlation — Building Features vs Entropy Features", fontsize=13, pad=14)
    ax.set_xlabel("Entropy features")
    ax.set_ylabel("Building features")
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)

    plt.tight_layout()
    plt.show()

## 8 — Pairplot of Building Features

Scatter matrix of the 7 building features.
Diagonal cells show per-city KDE distributions.
Off-diagonal cells show scatter plots coloured by city.

Use this to spot correlations, clusters, and outliers across the full feature space.

In [ ]:
pp_data = df[building_cols + ["code"]].dropna()

pair_grid = sns.pairplot(
    pp_data,
    hue="code",
    vars=building_cols,
    diag_kind="kde",
    plot_kws={"alpha": 0.35, "s": 12},
    diag_kws={"linewidth": 1.5},
    palette="tab10",
    corner=False,
)

pair_grid.figure.suptitle("Pairplot — 7 OSM Building Features (coloured by city)", y=1.01, fontsize=13)
pair_grid._legend.set_title("City")

# tighten tick labels
for ax in pair_grid.axes.flatten():
    if ax is not None:
        ax.tick_params(labelsize=8)
        ax.set_xlabel(ax.get_xlabel(), fontsize=9)
        ax.set_ylabel(ax.get_ylabel(), fontsize=9)

plt.tight_layout()
plt.show()

## 9 — Ranked Mean Entropy by City

Cities ranked from highest to lowest mean normalised street-orientation entropy.

Higher entropy → more irregular street network.
Lower entropy → more grid-like street network.

In [ ]:
# choose the primary entropy metric — prefer entropy_normalised if available
primary_entropy = next((c for c in ["entropy_normalised", "entropy_raw"] if c in df.columns), None)

if primary_entropy is None:
    print("No entropy column found — run 06_compute_entropy.py first.")
else:
    # compute mean entropy per city and rank descending (most irregular first)
    mean_entropy = (
        df.groupby("code")[primary_entropy]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )
    mean_entropy.columns = ["city", "mean_entropy"]
    mean_entropy["rank"] = range(1, len(mean_entropy) + 1)

    # colour bars by rank — darker = more irregular
    norm  = plt.Normalize(mean_entropy["mean_entropy"].min(),
                          mean_entropy["mean_entropy"].max())
    cmap  = plt.get_cmap("YlOrRd")
    colors = [cmap(norm(v)) for v in mean_entropy["mean_entropy"]]

    fig, ax = plt.subplots(figsize=(10, 5))

    bars = ax.barh(
        mean_entropy["city"],
        mean_entropy["mean_entropy"],
        color=colors,
        edgecolor="white",
        linewidth=0.6,
    )

    # annotate bars with the value
    for bar, val in zip(bars, mean_entropy["mean_entropy"]):
        ax.text(
            val + 0.002,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}",
            va="center",
            ha="left",
            fontsize=9,
        )

    ax.invert_yaxis()          # rank 1 at the top
    ax.set_xlabel(f"Mean {primary_entropy}")
    ax.set_title(f"Cities Ranked by Mean {primary_entropy} (most irregular → most grid-like)", fontsize=12)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.show()

    print("\nRanking table:")
    print(mean_entropy.to_string(index=False))

## 10 — Entropy Distribution per City (Ordered by Mean Slope, Flattest → Steepest)

Cities are ordered on the x-axis from the city with the lowest mean terrain slope (flattest) to the city with the highest mean terrain slope (steepest).

This lets us see whether more hilly cities also tend to have higher or lower street-direction entropy.

In [ ]:
if primary_entropy is None:
    print("No entropy column found — run 06_compute_entropy.py first.")
elif "mean_slope_deg" not in df.columns:
    print("mean_slope_deg not found — run 05_compute_terrain_poi_features.py first.")
else:
    # ── compute per-city statistics ───────────────────────────────────────────
    city_stats = df.groupby("code").agg(
        mean_slope=("mean_slope_deg", "mean"),
    ).reset_index()

    # sort cities flattest → steepest
    city_order = (
        city_stats
        .sort_values("mean_slope", ascending=True)["code"]
        .tolist()
    )

    # ── violin + box overlay ─────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(max(10, len(city_order) * 1.4), 6))

    sns.violinplot(
        data=df,
        x="code",
        y=primary_entropy,
        order=city_order,
        palette="YlOrRd",
        inner=None,
        linewidth=0.8,
        cut=0,
        ax=ax,
    )

    sns.boxplot(
        data=df,
        x="code",
        y=primary_entropy,
        order=city_order,
        width=0.18,
        color="white",
        fliersize=3,
        linewidth=1.2,
        ax=ax,
    )

    # annotate each city with its mean slope value
    for i, city in enumerate(city_order):
        slope_val = city_stats.loc[city_stats["code"] == city, "mean_slope"].values[0]
        ax.text(
            i,
            ax.get_ylim()[0] - (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.07,
            f"{slope_val:.1f}°",
            ha="center",
            va="top",
            fontsize=8.5,
            color="#555555",
        )

    # annotation label
    ax.text(
        -0.5,
        ax.get_ylim()[0] - (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.07,
        "mean slope:",
        ha="right",
        va="top",
        fontsize=8.5,
        color="#555555",
        style="italic",
    )

    ax.set_xlabel("City  (ordered flattest → steepest terrain)", labelpad=18)
    ax.set_ylabel(primary_entropy)
    ax.set_title(
        f"{primary_entropy} Distribution per City\n(ordered by mean terrain slope, flattest → steepest)",
        fontsize=12,
        pad=10,
    )
    ax.tick_params(axis="x", rotation=35)
    ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.show()

    # ── companion table ───────────────────────────────────────────────────────
    city_summary = df.groupby("code").agg(
        mean_slope   =("mean_slope_deg",  "mean"),
        mean_entropy =(primary_entropy,   "mean"),
        median_entropy=(primary_entropy,  "median"),
        std_entropy  =(primary_entropy,   "std"),
    ).loc[city_order].round(4)
    print("\nCity summary (flattest → steepest):")
    print(city_summary.to_string())